In [ ]:
from openai import OpenAI
import pandas as pd


client = OpenAI(
    base_url="",
    api_key="",
)


csv_path = r"G:\On the Naturalness of Agent-Generated Documentation\dataset\data\dev_agent_combined.csv"
df = pd.read_csv(csv_path)
df = df.dropna(subset=["doc_entropy", "doc_code_overlap", "doc_redundancy"])

sentiment_scores = []
groups = []

for index, row in df.iterrows():
    doc_text = row['doc_text'] 
    
    try:
      
        resp = client.chat.completions.create(
            model="gpt-oss-120b",
            messages=[{"role": "user", "content": f"Please perform Sentiment Classification task. Given the code documentation from functions, assign a sentiment label from ['negative', 'neutral', 'positive']. Return label only without any other text.\n\nCode documentation: {doc_text}"}],
            max_tokens=256,
            reasoning_effort="medium", 
        )


        label = resp.choices[0].message.content.lower().strip()
        sentiment_scores.append(label)
        groups.append(row['group'])
        print(f"Row {index} classified as: {label}")
        
    except Exception as e:
        print(f"Error processing row {index}: {e}")
        sentiment_scores.append("Error")
        groups.append(row['group'])


In [12]:
# make csv with 2 cols to save the data
df_sentiment = pd.DataFrame({'sentiment': sentiment_scores, 'group': groups})
df_sentiment.to_csv('sentiment_scores.csv', index=False)